In [1]:
import warnings
warnings.filterwarnings("ignore")

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import os
import json
import argparse
import sys
import gc
import torch
import torch.nn.functional as F
import torch.multiprocessing as mp

from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import numpy as np
import random
from typing import List
from re import split as rsplit


# ------------------------
# Utils
# ------------------------

def write_jsonl(data, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')


def read_jsonl(file_path):
    if not os.path.exists(file_path):
        print(f"Warning: Dataset file not found at {file_path}")
        return []
    with open(file_path, 'r', encoding='utf-8') as f:
        return [json.loads(line.strip()) for line in f if line.strip()]


def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.npu.is_available():
        torch.npu.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.set_float32_matmul_precision("high")


def _clear_npu():
    if torch.npu.is_available():
        try:
            torch.npu.synchronize()
        except Exception:
            pass
        torch.npu.empty_cache()
        try:
            torch.npu.ipc_collect()  # Inter-Process Communication
        except Exception:
            pass
    gc.collect()  # Garbage Collection


# ------------------------
# Confidence helpers
# ------------------------

def _summarize_selected_logprobs(selected_logprobs: torch.Tensor, policy: str = 'avg2') -> float:
    if selected_logprobs is None or selected_logprobs.numel() == 0:
        return 0.0
    probs = torch.exp(selected_logprobs)
    if policy == 'min':
        return probs.min().item()
    elif policy == 'avg1':
        return probs.mean().item()
    return torch.exp(selected_logprobs.mean()).item()


def compute_token_logprobs_streaming(model, tokenizer, prompt_ids: torch.Tensor, gen_ids: torch.Tensor,
                                     device: torch.device, score_dtype: str = 'bf16') -> torch.Tensor:
    assert prompt_ids.ndim == 2 and prompt_ids.size(0) == 1
    assert gen_ids.ndim == 1

    dtype_map = {'bf16': torch.bfloat16, 'fp16': torch.float16, 'fp32': torch.float32}
    amp_dtype = dtype_map.get(score_dtype, torch.bfloat16)

    logps = []
    with torch.inference_mode(), torch.npu.amp.autocast(enabled=(device.type == 'npu'), dtype=amp_dtype):
        out = model(prompt_ids.to(device), use_cache=True, return_dict=True)
        past = out.past_key_values
        logits = out.logits[:, -1, :]
        # TODO: what is calculated here is the probability of the sampled token, rather than the maximum probability
        logprob0 = F.log_softmax(logits, dim=-1)[0, gen_ids[0].to(device)].item()
        logps.append(logprob0)

        if gen_ids.numel() > 1:
            prev = gen_ids[0].view(1, 1).to(device)
            for t in range(1, gen_ids.numel()):
                out = model(prev, use_cache=True, past_key_values=past, return_dict=True)
                past = out.past_key_values
                logits = out.logits[:, -1, :]
                lp = F.log_softmax(logits, dim=-1)[0, gen_ids[t].to(device)].item()
                logps.append(lp)
                prev = gen_ids[t].view(1, 1).to(device)

    return torch.tensor(logps, dtype=torch.float32)


# ------------------------
# Model / tokenizer
# ------------------------

def load_model_and_tokenizer_single_npu(args, device):
    tokenizer = AutoTokenizer.from_pretrained(
        args.model_name_or_path,
        use_fast=False,
        trust_remote_code=args.trust_remote_code,
        local_files_only=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        args.model_name_or_path,
        trust_remote_code=args.trust_remote_code,
        torch_dtype="auto",
        device_map="npu",
        local_files_only=True
    )

    model.to(device)
    model.eval()

    return model, tokenizer


# ------------------------
# Hidden-state saving (修复版)
# ------------------------

def save_pangu_think_split_tokens_only(model, tokenizer, input_ids, full_text, save_path,
                                       hs_device: str = 'auto'):
    import os as _os

    ids_flat = input_ids[0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])  # TODO: change to tokenizer.decode()?
    think_ids = tokenizer.encode("[unused17]", add_special_tokens=False)  # ! adapt to pangu

    def _find_subseq(seq, subseq):
        if not subseq:
            return None
        L, M = len(seq), len(subseq)
        for s in range(0, L - M + 1):
            if seq[s:s + M] == subseq:
                return s
        return None

    pos = _find_subseq(ids_flat, think_ids)
    think_end_idx = pos if pos is not None else len(ids_flat)

    step_positions = []
    for i in range(think_end_idx - 1):
        tok = tokens[i]
        if "\n\n" in tok:  # ! adpat to pangu
            step_positions.append(i + 1)  # TODO: why + 1 here?

    hidden_dict = {}

    def _forward_on(device):
        with torch.inference_mode():
            outputs = model(input_ids.to(device), output_hidden_states=True, use_cache=False, return_dict=True)
        hidden_states = outputs.hidden_states
        for layer_id, layer_h in enumerate(hidden_states):
            h = layer_h.squeeze(0)
            if len(step_positions) > 0:
                idx = torch.tensor(step_positions, dtype=torch.long, device=h.device)  # TODO: extract the next token of /n/n?
                step_h = h.index_select(dim=0, index=idx).to('cpu', non_blocking=True)
            else:
                step_h = torch.empty((0, h.shape[1]), dtype=h.dtype)
            hidden_dict[layer_id] = step_h  # simplify the dict structure

    orig_device = next(model.parameters()).device
    try:
        if hs_device in ('auto', 'npu') and torch.npu.is_available():
            _forward_on(orig_device)
        else:
            if orig_device.type != 'cpu':
                model.to('cpu')
            _forward_on(torch.device('cpu'))
    except torch.npu.OutOfMemoryError:
        print("[hs][npu OOM] falling back to CPU for hidden-state dump...")
        _clear_npu()  # TODO: clean up NPU cache here actually doesn't have much effect?
        if hs_device == 'npu':
            raise
        model.to('cpu')
        _forward_on(torch.device('cpu'))
    finally:
        if next(model.parameters()).device != orig_device:
            model.to(orig_device)

    _os.makedirs(_os.path.dirname(save_path), exist_ok=True)
    tmp_path = save_path + ".tmp"
    torch.save(hidden_dict, tmp_path)
    _os.replace(tmp_path, save_path)


# ------------------------
# Shard helpers
# ------------------------

def load_existing_indices(shard_file):
    existing_idx = set()
    existing_q = set()
    num_lines = 0
    if os.path.exists(shard_file):
        with open(shard_file, 'r', encoding='utf-8') as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                num_lines += 1
                try:
                    obj = json.loads(s)
                    if "idx" in obj:
                        existing_idx.add(int(obj["idx"]))
                    if "question" in obj:
                        existing_q.add(obj["question"])
                except Exception:
                    continue
    return existing_idx, existing_q, num_lines


def _read_jsonl_map_by_idx(file_path):
    m = {}
    if not os.path.exists(file_path):
        return m
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
                if "idx" in obj:
                    m[int(obj["idx"])] = obj
            except Exception:
                continue
    return m


def _reconstruct_full_text(tokenizer, sys_prompt, question_text, response_text):
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": question_text}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt + (response_text or "")

In [2]:
parser = argparse.ArgumentParser()
parser.add_argument('--model_name_or_path', type=str, required=True)
parser.add_argument('--dataset_dir', type=str, required=True)
parser.add_argument('--output_path', type=str, required=True)
parser.add_argument('--dataset', type=str, required=True)
parser.add_argument('--temperature', type=float, default=0.7)
parser.add_argument('--top_p', type=float, default=0.95)
parser.add_argument('--max_generated_tokens', type=int, default=512)
parser.add_argument('--trust_remote_code', action='store_true')
parser.add_argument('--num_npus', type=int, default=1)
parser.add_argument('--hs_device', type=str, default='auto', choices=['auto', 'npu', 'cpu'])
parser.add_argument('--score_dtype', type=str, default='bf16', choices=['bf16', 'fp16', 'fp32'])

_StoreAction(option_strings=['--score_dtype'], dest='score_dtype', nargs=None, const=None, default='bf16', type=<class 'str'>, choices=['bf16', 'fp16', 'fp32'], required=False, help=None, metavar=None)

# Debug

In [3]:
args = parser.parse_args(args=[
    "--model_name_or_path", "/home/ma-user/work/dataset/models_yulin/openPangu-Embedded-7B-V1.1",
    "--dataset_dir", "/home/ma-user/work/algorithm/Rebalance/Data",
    "--dataset", "Math_Math",
    "--output_path", "/home/ma-user/work/dataset/outputs_yulin/test",
    "--max_generated_tokens", "16000",
    "--num_npus", "1",
    "--trust_remote_code"
])

In [8]:
rank = 0
world_size = 1
    
# set_seeds(42 + rank)
if torch.npu.is_available():
    torch.npu.set_device(rank)
device = torch.device(f"npu:{rank}" if torch.npu.is_available() else "cpu")

# Use train.jsonl and limit to 500 samples for Math_Math; otherwise use test.jsonl
dataset_file = 'train.jsonl' if args.dataset == 'Math_Math' else 'test.jsonl'
dataset_path = os.path.join(args.dataset_dir, args.dataset, dataset_file)
questions = read_jsonl(dataset_path)

if args.dataset == 'Math_Math':
    questions = questions[26:27]

model_basename = os.path.basename(os.path.normpath(args.model_name_or_path))
output_dir = os.path.join(args.output_path, model_basename, args.dataset)
os.makedirs(output_dir, exist_ok=True)
base_name = f'origin_temp{args.temperature}_maxlen{args.max_generated_tokens}'
shard_file = os.path.join(output_dir, f'{base_name}.shard{rank:03d}.jsonl')

existing_idx, existing_q, num_lines = load_existing_indices(shard_file)
existing_map = _read_jsonl_map_by_idx(shard_file)

model, tokenizer = load_model_and_tokenizer_single_npu(args, device)



/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.10/site-packages/torch_npu/contrib/transfer_to_npu.py:295: ImportWarning: 
    *************************************************************************************************************
    The torch.Tensor.cuda and torch.nn.Module.cuda are replaced with torch.Tensor.npu and torch.nn.Module.npu now..
    The torch.cuda.DoubleTensor is replaced with torch.npu.FloatTensor cause the double type is not supported now..
    The backend in torch.distributed.init_process_group set to hccl now..
    The torch.cuda.* and torch.cuda.amp.* are replaced with torch.npu.* and torch.npu.amp.* now..
    The device parameters have been replaced with npu in the function below:
    torch.logspace, torch.randint, torch.hann_window, torch.rand, torch.full_like, torch.ones_like, torch.rand_like, torch.randperm, torch.arange, torch.frombuffer, torch.normal, torch._empty_per_channel_affine_quantized, torch.empty_strided, torch.empty_like, torch.scalar_t

[INFO] torch_npu detected. Using NPU fused infer attention.


Loading checkpoint shards: 100%|██████████| 4/4 [00:13<00:00,  3.43s/it]


In [17]:
my_indices = [i for i in range(len(questions)) if (i % world_size) == rank]

print(f"[rank {rank}] world_size={world_size}")
print(f"[rank {rank}] shard_file = {shard_file}")
print(f"[rank {rank}] loaded existing: idx={len(existing_idx)}, question={len(existing_q)}, lines={num_lines}")
print(f"[rank {rank}] will process {len(my_indices)} items")

pbar = tqdm(total=len(my_indices), desc=f"Rank {rank} DP Inference", position=rank, leave=True)

sys_prompt = "Please reason step by step, and put your final answer within \\boxed{}."  # unify prompt here

i = 0
q = questions[i]
hidden_save_path = os.path.join(output_dir, f"hidden_{i:03}.pt")

# if (i in existing_idx) or (q.get("problem") in existing_q):
#     if not os.path.exists(hidden_save_path):
#         try:
#             entry = existing_map.get(i)
#             if entry and entry.get("generated_responses"):
#                 response_text = entry["generated_responses"][0] if entry["generated_responses"] else ""
#                 full_text = _reconstruct_full_text(tokenizer, sys_prompt, q.get("problem", ""), response_text)
#                 full_inputs = tokenizer([full_text], return_tensors="pt").to(device)
#                 save_pangu_think_split_tokens_only(
#                     model, tokenizer, full_inputs.input_ids.to(device), full_text, hidden_save_path,
#                     hs_device=args.hs_device
#                 )
#                 print(f"[rank {rank}] rescued hidden for idx={i}")
#                 del full_inputs
#                 _clear_npu()
#         except Exception as e_rescue:
#             print(f"[WARN][rank {rank}] rescue hidden failed at idx={i}: {e_rescue}")
#             _clear_npu()
#     pbar.update(1)
#     continue

messages = [
    {"role": "system", "content": sys_prompt},
    {"role": "user", "content": q['problem']}
]


[rank 0] world_size=1
[rank 0] shard_file = /home/ma-user/work/dataset/outputs_yulin/test/openPangu-Embedded-7B-V1.1/Math_Math/origin_temp0.7_maxlen16000.shard000.jsonl
[rank 0] loaded existing: idx=0, question=0, lines=0
[rank 0] will process 1 items


Rank 0 DP Inference:   0%|          | 0/1 [01:07<?, ?it/s]


In [34]:
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to(device)

try:
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            do_sample=True,
            temperature=args.temperature,
            top_p=args.top_p,
            max_new_tokens=args.max_generated_tokens,
            return_dict_in_generate=True,
            eos_token_id=45892
        )
except torch.npu.OutOfMemoryError:
    _clear_npu()
    print(f"[OOM][rank {rank}] idx={i} : {q['problem'][:80]}... skipping.")
    pbar.update(1)
    # continue
except Exception as e:
    print(f"[ERROR][rank {rank}] idx={i} : {e}")
    pbar.update(1)
    # continue


[W compiler_depend.ts:52] Warning: Version:  is invalid. (function operator())


In [58]:
response_text = tokenizer.decode(
    output.sequences[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)
full_text = prompt + response_text
gen_token_ids = output.sequences[0][inputs.input_ids.shape[1]:].detach().cpu()

In [61]:
try:
    gen_logps = compute_token_logprobs_streaming(
        model, tokenizer,
        prompt_ids=inputs.input_ids,
        gen_ids=gen_token_ids,
        device=device,
        score_dtype=args.score_dtype,
    )
except torch.npu.OutOfMemoryError:
    print(f"[OOM][rank {rank}] logprob pass at idx={i}; skipping confidences.")
    gen_logps = torch.empty(0)
except Exception as e:
    print(f"[WARN][rank {rank}] logprob pass failed at idx={i}: {e}")
    gen_logps = torch.empty(0)

In [97]:
confidences = []
start = 0
sum = 0
for segment in text_segments:
    seg_ids = tokenizer(segment, add_special_tokens=False).input_ids
    sum += len(seg_ids)
    end = start + len(seg_ids)
    if gen_logps.numel() > 0 and end > start and end <= gen_logps.numel():
        seg_logps = gen_logps[start:end]
        conf = _summarize_selected_logprobs(seg_logps, policy='avg2')
        confidences.append(conf)
    elif end > start:
        confidences.append(0.0)
    start = end

In [128]:
text_before_think = response_text.split('[unused17]')[0]
tokens_before_think = tokenizer.tokenize(text_before_think)

token_segments = []
current_segment = []
for token in tokens_before_think:
    if "\n\n" in token:
        if current_segment:
            token_segments.append(current_segment)
            current_segment = []
    else:
        current_segment.append(token)

# add the last segment (if exists)
if current_segment:
    token_segments.append(current_segment)

In [156]:
# test
a = "[unused16]I need to find \\(\\binom{25}{5}\\) given \\(\\binom{23}{3} = 1771\\), \\(\\binom{23}{4} = 8855\\), and \\(\\binom{23}{5} = 33649\\). I recall that binomial coefficients can be related through identities like Pascal's identity, which states that \\(\\binom{n}{k} = \\binom{n-1}{k-1} + \\binom{n-1}{k}\\).\n\nSince I need \\(\\binom{25}{5}\\), and I have values for n=23, I should build up to 25 using Pascal's identity repeatedly. Let me see how I can get from 23 to 25.\n\nI know that \\(\\binom{25}{5} = \\binom{24}{4} + \\binom{24}{5}\\), by Pascal's identity. Then, I can express \\(\\binom{24}{4}\\) and \\(\\binom{24}{5}\\) in terms of n=23.\n\nSpecifically, \\(\\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4}\\), and \\(\\binom{24}{5} = \\binom{23}{4} + \\binom{23}{5}\\).\n\nThat's perfect! So, if I let \\(A = \\binom{23}{3}\\), \\(B = \\binom{23}{4}\\), and \\(C = \\binom{23}{5}\\), which are given as 1771, 8855, and 33649 respectively.\n\nThen, \\(\\binom{24}{4} = A + B = 1771 + 8855\\)\n\nCalculating that: 1771 + 8855. 1771 + 8000 = 9771, plus 855 is 9771 + 855. Better: 1771 + 8855.\n\n1771 + 8855: 71 + 55 = 126, carry over 1; 7 + 85 = 92, plus 1 is 93? No.\n\nAdd step by step: 1771 + 8855. Start from right: 1 + 5 = 6. 7 + 5 = 12, write 2, carry 1. 7 + 8 = 15, plus 1 is 16, write 6, carry 1. 1 + 8 = 9, plus 1 is 10. So, 10556? That doesn't seem right.\n\n1771 + 8855. 1771 + 8000 = 9771, then 9771 + 855. 9771 + 800 = 10571, then 10571 + 55 = 10556. But earlier I thought it should be more, but let me verify.\n\nI could compute numerically: 1771 + 8855. 1700 + 8800 = 10500, 71 + 55 = 126, total 10500 + 126 = 10626. I think I messed up.\n\n1771 is 1700 + 71, but better: 1771 + 8855.\n\nAdd: 1771\n\n+8855\n\n------\n\n1+5=6\n\n7+5=12, write 2, carry 1\n\n7+8=15, plus 1 is 16, write 6, carry 1\n\n1+8=9, plus 1 is 10? Wait, the thousands: 1 (from 1771) and 8 (from 8855), but 1771 has 1 thousand, 8855 has 8 thousands? No.\n\n1771 is 1 thousand, 7 hundreds, 7 tens, 1 unit.\n\n8855 is 8 thousands, 8 hundreds, 5 tens, 5 units.\n\nSo:\n\nUnits: 1 + 5 = 6\n\nTens: 7 + 5 = 12, write 2, carry 1\n\nHundreds: 7 + 8 = 15, plus carry 1 = 16, write 6, carry 1\n\nThousands: 1 + 8 = 9, plus carry 1 = 10, write 0, carry 1\n\nTen thousands: carry 1, so 10556.\n\nBut earlier I thought 10556, but that can't be right because \\(\\binom{23}{4}\\) is already 8855, and adding \\(\\binom{23}{3}=1771\\) should give something around 10,000, but 10556 seems plausible. Let me confirm with actual values if possible, but I don't have a calculator, but I think it's correct.\n\nActually, I recall that \\(\\binom{23}{4} = \\frac{23!}{4!19!}\\), but I don't need to compute it now. Anyway, so \\(\\binom{24}{4} = 1771 + 8855 = 10556\\).\n\n1771 + 8855: 1700 + 8855 = 10555, plus 71 is 10556. Yes.\n\nNow, \\(\\binom{24}{5} = \\binom{23}{4} + \\binom{23}{5} = B + C = 8855 + 33649\\).\n\nCompute that: 8855 + 33649.\n\n8855 + 33000 = 41855, plus 649 = 41855 + 600 = 42455, plus 49 = 42455 + 50 - 1 = 42505 - 1 = 42494? Better:\n\n8855 + 33649.\n\nAdd: 8855\n\n+33649\n\n------\n\n5+9=14, write 4, carry 1\n\n5+4=9, plus 1=10, write 0, carry 1\n\n8+6=14, plus 1=15, write 5, carry 1\n\n8+3=11, plus 1=12, write 2, carry 1\n\n3 + carry 1 = 4? Wait, 33649 has 5 digits, 8855 has 4, so:\n\nAlign:\n\n  8855\n\n+33649\n\n------\n\nStart from right:\n\n5+9=14, write 4, carry 1\n\n5+4=9, plus 1=10, write 0, carry 1\n\n8+6=14, plus 1=15, write 5, carry 1\n\n8+3=11, plus 1=12, write 2, carry 1\n\n3 + carry 1 = 4? But 33649 has 3 in ten thousands, so it's 33649, so ten thousands is 3, thousands is 3, etc.\n\n8855 is 08855, so:\n\n   08855\n\n+ 33649\n\n------\n\nUnits: 5+9=14, write 4, carry 1\n\nTens: 5+4=9, plus 1=10, write 0, carry 1\n\nHundreds: 8+6=14, plus 1=15, write 5, carry 1\n\nThousands: 8+3=11, plus 1=12, write 2, carry 1\n\nTen thousands: 0+3=3, plus 1=4, so 42494.\n\nYes, so \\(\\binom{24}{5} = 8855 + 33649 = 42494\\).\n\nNow, back to \\(\\binom{25}{5} = \\binom{24}{4} + \\binom{24}{5} = 10556 + 42494\\).\n\nAdd those: 10556 + 42494.\n\n10556\n\n+42494\n\n------\n\n6+4=10, write 0, carry 1\n\n5+9=14, plus 1=15, write 5, carry 1\n\n5+4=9, plus 1=10, write 0, carry 1\n\n0+2=2, plus 1=3? Wait, thousands: 10556 has 10 thousands? Better to write as:\n\n  10556\n\n+42494\n\n------\n\nUnits: 6+4=10, write 0, carry 1\n\nTens: 5+9=14, plus 1=15, write 5, carry 1\n\nHundreds: 5+4=9, plus 1=10, write 0, carry 1\n\nThousands: 0+2=2, plus 1=3? But 10556 has 10 thousands? 10556 is 1 ten thousand, 0 thousands? No.\n\n10556: 1 (ten thousands), 0 (thousands), 5 (hundreds), 5 (tens), 6 (units). Standard notation: 10556 is 10,556.\n\nSimilarly, 42494 is 42,494.\n\nSo:\n\nThousands: 10556 has 10 thousands, 42494 has 42 thousands? Better to add digit by digit.\n\nWrite:\n\n   10556\n\n+  42494\n\n--------\n\nStart from right:\n\n6 + 4 = 10, write 0, carry 1\n\n5 + 9 + 1 (carry) = 15, write 5, carry 1\n\n5 + 4 + 1 = 10, write 0, carry 1\n\n0 + 2 + 1 = 3, write 3? But 10556 has a 1 in ten thousands, and 42494 has 4 in ten thousands? I missed the ten thousands.\n\n10556 is 1*10000 + 0*1000 + 5*100 + 5*10 + 6\n\n42494 is 4*10000 + 2*1000 + 4*100 + 9*10 + 4\n\nSo:\n\nTen thousands: 1 + 4 = 5? But with carries.\n\nUnits: 6 + 4 = 10, write 0, carry 1\n\nTens: 5 + 9 = 14, plus carry 1 = 15, write 5, carry 1\n\nHundreds: 5 + 4 = 9, plus carry 1 = 10, write 0, carry 1\n\nThousands: 0 + 2 = 2, plus carry 1 = 3, write 3\n\nTen thousands: 1 + 4 = 5, write 5? But no carry, so 53,050? But that can't be right because 10556 + 42494 should be around 53,000 something.\n\n10556 + 42494 = 53,050? Let me compute: 10000 + 42000 = 52,000; 556 + 494 = 1050; total 52,000 + 1050 = 53,050. Oh, so 53,050.\n\nBut earlier I thought it should be more, but numerically it's correct. 10556 + 42494 = 53050? 10000 + 42000 = 52000, 556 + 494 = 1050, total 53050. But I wrote 53,050, which is the same as 53050? No, 53,050 is fifty-three thousand fifty, which is 53050, yes.\n\nIn my addition, I got 53050, but let me confirm: thousands: 0 (from 10556) + 2 (from 42494) = 2, but with carry 1 from hundreds, so 2+1=3, and ten thousands 1+4=5, so 53,050? 53,050 is fifty-three thousand fifty, but in numerical form it's 53050, but that's correct? 10556 + 42494.\n\n10556 + 40000 = 50556, plus 2494 = 50556 + 2000 = 52556, plus 494 = 52556 + 500 - 6 = 53056 - 6 = 53050? 52556 + 494: 556 + 494 = 1050, so 52556 + 494 = 52556 + 500 - 6 = 53056 - 6 = 53050. But earlier I said 53050, but in the addition I wrote 53,050, which is the same as 53050, but I think I missed a digit.\n\n10556 has five digits, 42494 has five digits, sum should be five or six digits. 10556 is about 10k, 42494 is about 42k, sum about 52k, but 53050 is 53k, close.\n\nBut let me add: 10556 + 42494.\n\n6+4=10, write 0, carry 1\n\n5+9=14, plus 1=15, write 5, carry 1\n\n5+4=9, plus 1=10, write 0, carry 1\n\n0+2=2, plus 1=3, write 3\n\n1+4=5, write 5. So 53050.\n\nBut 10556 + 42494: let me calculate numerically: 10000 + 42000 = 52000, but 10556 is 556 more than 10000, 42494 is 494 more than 42000? Better: 10556 + 42494 = (10000 + 42000) + (556 + 494) = 52000 + 1050 = 53050. Yes, correct.\n\nSo \\(\\binom{25}{5} = 53050\\).\n\nBut I feel like this might be too straightforward, and I recall that \\(\\binom{25}{5}\\) should be a larger number. Let me verify with a smaller example or something.\n\nI know that \\(\\binom{25}{5} = \\frac{25 \\times 24 \\times 23 \\times 22 \\times 21}{5 \\times 4 \\times 3 \\times 2 \\times 1} = \\frac{25 \\times 24 \\times 23 \\times 22 \\times 21}{120}\\).\n\nFirst, compute numerator: 25 × 24 = 600\n\n600 × 23 = 13800\n\n13800 × 22 = 13800 × 20 + 13800 × 2 = 276000 + 27600 = 303600\n\n303600 × 21 = 303600 × 20 + 303600 × 1 = 6072000 + 303600 = 6375600\n\nNow divide by 120: 6375600 / 120.\n\nFirst, /10 = 637560, then /12.\n\n120 = 12 × 10, but better: 6375600 / 120.\n\nDivide by 10: 637560, then divide by 12.\n\n637560 / 12.\n\n12 × 53130 = ? 12 × 50000 = 600000, subtract 637560 - 600000 = 37560\n\n12 × 3131 = 12 × 3000 = 36000, 12 × 131 = 12 × 130 = 1560, 12 × 1 = 12, so 1560 + 12 = 1572, total 36000 + 1572 = 37572? Messy.\n\n637560 / 12.\n\nDivide: 12 into 63 is 5, 5×12=60, subtract 63-60=3\n\nBring down 5, 35. 12×2=24, 35-24=11\n\nBring down 7, 117. 12×9=108, 117-108=9\n\nBring down 6, 96. 12×8=96, 0.\n\nBring down 0, 0. 12×0=0. So 53130? But that's for 637560 / 12, but I divided by 10 already? I think I confused.\n\nI have 6375600 / 120.\n\nFirst, /120 = /(12×10) = (6375600 / 10) / 12 = 637560 / 12.\n\nNow, 637560 ÷ 12.\n\n12 × 53130 = ? As above: 12 × 50000 = 600000\n\nSubtract: 637560 - 600000 = 37560\n\n12 × 3100 = 37200? Better: 12 × 53130.\n\nFrom earlier division: 12 into 63 is 5, 5×12=60, subtract 63-60=3\n\nBring down 5, 35. 12×2=24, subtract 35-24=11\n\nBring down 7, 117. 12×9=108, subtract 117-108=9\n\nBring down 6, 96. 12×8=96, subtract 0.\n\nBring down 0, 0. 12×0=0. So 53130.\n\nBut 53130, but earlier I thought \\(\\binom{25}{5}\\) should be larger. 53130? But I calculated numerator 6375600, divided by 120.\n\n6375600 / 120.\n\n120 × 53130 = 120 × 50000 = 6,000,000; 120 × 3130 = 120 × 3000 = 360,000; 120 × 130 = 15,600; total 6,000,000 + 360,000 = 6,360,000 + 15,600 = 6,375,600. But I have 6,375,600? Earlier I had 6,375,600? 6375600 is 6,375,600? 6375600 is 6,375,600? 6 million, 375 thousand, 600, but I wrote 6375600, which is 6,375,600, but in calculation I had 6375600, which is correct? Earlier I computed 25×24×23×22×21.\n\n25×24=600\n\n600×23=13,800\n\n13,800×22: 13,800×20=276,000; 13,800×2=27,600; sum 276,000 + 27,600 = 303,600\n\n303,600×21: 303,600×20=6,072,000; 303,600×1=303,600; sum 6,072,000 + 303,600 = 6,375,600. Yes.\n\nNow /120: 6,375,600 / 120.\n\n120 = 12 × 10, so /10 = 637,560; then /12.\n\n637,560 / 12.\n\n12 × 53,130 = ? 12 × 50,000 = 600,000; 12 × 3,000 = 36,000; total 636,000; subtract from 637,560: 637,560 - 636,000 = 1,560.\n\n12 × 130 = 1,560. So total 50,000 + 3,000 + 130 = 53,130. Yes.\n\nBut earlier I got 53,050, which is different. What? I think I made a mistake in the addition.\n\nI had \\(\\binom{24}{4} = 1771 + 8855 = 10,556\\)\n\n\\(\\binom{24}{5} = 8855 + 33649 = 42,494\\)\n\nThen sum 10,556 + 42,494 = ?\n\n10,556 + 42,494.\n\nAdd: 10,556 + 40,000 = 50,556; plus 2,494 = 50,556 + 2,500 - 6 = 53,056 - 6 = 53,050? 50,556 + 2,494.\n\n50,556 + 2,000 = 52,556; plus 494 = 52,556 + 500 - 6 = 53,056 - 6 = 53,050. Yes.\n\nBut from direct calculation, I got 53,130. Inconsistency here.\n\nDirect calculation gave 53,130, but my method gave 53,050. So I must have made a mistake.\n\nWhere did I go wrong? Let me check the values given.\n\nGiven \\(\\binom{23}{3} = 1771\\), \\(\\binom{23}{4} = 8855\\), \\(\\binom{23}{5} = 33649\\).\n\nI know that \\(\\binom{23}{3} = \\frac{23×22×21}{6} = \\frac{23×22×21}{6}\\).\n\n23×22 = 506, 506×21 = 506×20 + 506×1 = 10,120 + 506 = 10,626; divide by 6: 10,626 / 6 = 1771. Yes, correct.\n\n\\(\\binom{23}{4} = \\frac{23×22×21×20}{24} = \\frac{23×22×21×20}{24}\\).\n\nFirst, 23×22=506, 506×21=10,626, 10,626×20=212,520; divide by 24.\n\n24 = 4×6, but 212,520 / 24.\n\n24 × 8855? 24 × 8000 = 192,000; subtract 212,520 - 192,000 = 20,520; 24 × 855 = 24×800=19,200; 24×55=1,320; total 19,200+1,320=20,520. So 8000+855=8855. Yes.\n\n\\(\\binom{23}{5} = \\frac{23×22×21×20×19}{120}\\).\n\nFirst, 23×22=506, 506×21=10,626, 10,626×20=212,520, 212,520×19.\n\n212,520×20 = 4,250,400 minus 212,520 = 4,037,880? Better: 212,520 × 10 = 2,125,200; 212,520 × 9 = 1,912,680; total 2,125,200 + 1,912,680 = 4,037,880.\n\nNow divide by 120: 4,037,880 / 120.\n\nFirst, /10 = 403,788; then /12.\n\n403,788 / 12.\n\n12 × 33,649 = ? 12 × 30,000 = 360,000; 12 × 3,649.\n\n12 × 3,000 = 36,000; 12 × 649 = 12×600=7,200; 12×49=588; total 7,200+588=7,788; so total 36,000 + 7,788 = 43,788; then total for 33,649: 30,000 + 3,649 = 33,649? 12 × 33,649.\n\nBetter: 12 × 33,000 = 396,000; 12 × 649 = as above 7,788; total 396,000 + 7,788 = 403,788. Yes, so 33,649. Correct.\n\nNow, \\(\\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4} = 1771 + 8855 = 10,556\\).\n\nBut what is actual \\(\\binom{24}{4}\\)? \\(\\frac{24×23×22×21}{24} = \\frac{24}{24} × \\frac{23×22×21}{6} = 1 × \\binom{23}{3}? No.\n\n\\(\\binom{24}{4} = \\frac{24×23×22×21}{4×3×2×1} = \\frac{24}{4} × \\frac{23×22×21}{6} = 6 × \\binom{23}{3}? \\binom{23}{3} = \\frac{23×22×21}{6}\\), so \\(\\binom{24}{4} = \\frac{24×23×22×21}{24} = \\frac{24}{24} × 23×22×21 / 1? No.\n\nStandard: \\(\\binom{24}{4} = \\frac{24×23×22×21}{4×3×2×1} = \\frac{24}{4} × \\frac{23×22×21}{3×2×1} = 6 × \\binom{23}{3}\\), since \\binom{23}{3} = \\frac{23×22×21}{6}, so yes, 6 × 1771 = 10,626.\n\nBut I calculated 1771 + 8855 = 10,556, which is different from 10,626. Ah! I see, I made a mistake here.\n\n\\(\\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4}\\), but is that correct? Pascal's identity: \\(\\binom{n}{k} = \\binom{n-1}{k-1} + \\binom{n-1}{k}\\).\n\nSo for n=24, k=4: \\(\\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4}\\). Yes, that's correct.\n\nBut numerically, \\binom{23}{3} = 1771, \\binom{23}{4} = 8855, sum 1771 + 8855 = 10,556.\n\nBut from direct calculation, it should be 6 × 1771 = 10,626. 10,556 vs 10,626, difference of 70, so mistake.\n\n1771 + 8855: 1771 + 8000 = 9771, 9771 + 855 = 9771 + 800 = 10571, 10571 + 55 = 10556, yes.\n\nBut 6 × 1771: 6 × 1700 = 10200, 6 × 71 = 426, total 10200 + 426 = 10626, yes.\n\nBut according to Pascal, it should be equal, but 1771 + 8855 = 10556, while \\binom{24}{4} should be 10626, but 10556 ≠ 10626. What's wrong?\n\nI think I confused the identity. Pascal's identity is \\binom{n}{k} = \\binom{n-1}{k-1} + \\binom{n-1}{k}.\n\nFor \\binom{24}{4}, n=24, k=4, so \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4}, yes.\n\nBut \\binom{23}{3} = 1771, \\binom{23}{4} = 8855, sum 10556.\n\nBut \\binom{24}{4} = \\frac{24!}{4!20!} = \\frac{24×23×22×21×20!}{24 × 20!} = \\frac{24×23×22×21}{24} = 23×22×21 / 1? No.\n\n\\frac{24×23×22×21}{4×3×2×1} = \\frac{24}{4} × \\frac{23×22×21}{6} = 6 × \\binom{23}{3} = 6 × 1771 = 10626.\n\nBut 1771 + 8855 = 10556, which is not 10626. So why the discrepancy?\n\nI see the issue. \\binom{23}{4} is not needed for this. But according to identity, it should be equal, but it's not. Unless I miscalculated the sum or the values.\n\n\\binom{23}{4} = 8855, \\binom{23}{3} = 1771, 1771 + 8855 = 10556.\n\nBut from direct, \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4} should be true, but numerically it's not matching, so perhaps the given \\binom{23}{4} is wrong? But I calculated it as 8855, and it matched when I computed \\binom{23}{4} directly.\n\nLet me calculate \\binom{24}{4} directly: \\frac{24×23×22×21}{24} = \\frac{24}{24} × 23×22×21 / 1? No.\n\n4! = 24, so \\frac{24×23×22×21}{24} = 23×22×21 / 1? That's not right.\n\n\\frac{24×23×22×21}{4×3×2×1} = \\frac{24}{4} × \\frac{23×22×21}{3×2×1} = 6 × \\frac{23×22×21}{6} = 6 × \\binom{23}{3} = 6 × 1771 = 10626.\n\n\\binom{23}{3} = \\frac{23×22×21}{6} = \\frac{23×22×21}{6}, and 23×22×21 = 23×22=506, 506×21=10626, /6 = 1771, yes.\n\nNow \\binom{23}{4} = \\frac{23×22×21×20}{24} = \\frac{23×22×21×20}{24}.\n\n23×22×21 = 10626, times 20 = 212520, divided by 24.\n\n212520 / 24: 24×8855 = as before, 24×8000=192000, 24×855=24×800=19200, 24×55=1320, 19200+1320=20520, total 192000+20520=212520, yes 8855.\n\nNow for \\binom{24}{4}, it should be \\binom{23}{3} + \\binom{23}{4} = 1771 + 8855 = 10556.\n\nBut 6 × 1771 = 10626, and 10626 ≠ 10556, so the identity is not holding? But it must.\n\nI think I know: Pascal's identity is \\binom{n}{k} = \\binom{n-1}{k-1} + \\binom{n-1}{k}, so for n=24, k=4, \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4}, but \\binom{23}{3} is for k-1=3, n-1=23, yes.\n\nBut numerically, 1771 + 8855 = 10556, but \\binom{24}{4} = 10626, so difference of 70.\n\nPerhaps I have the wrong \\binom{23}{4}. Let me calculate \\binom{24}{4} directly.\n\n\\binom{24}{4} = \\frac{24×23×22×21}{4×3×2×1} = \\frac{24}{4} × \\frac{23×22×21}{6} = 6 × \\frac{23×22×21}{6} = 6 × 1771 = 10626, as before.\n\nNow \\binom{23}{4} = \\frac{23×22×21×20}{24} = \\frac{23×22×21×20}{24}.\n\n23×22×21 = 10626, as above, times 20 = 212520, divided by 24.\n\n212520 / 24: 24 × 8850 = 24×8000=192000, 24×850=20400, total 212400, then 212520 - 212400 = 120, 24×5=120, so 8850+5=8855, yes.\n\nBut 1771 + 8855 = 10556, while 10626, so why?\n\nI see the mistake! When I calculate \\binom{24}{4}, it is \\binom{23}{3} + \\binom{23}{4}, but \\binom{23}{3} is correct, but \\binom{23}{4} is for the same n-1, but in the identity, it's correct, but numerically it's not adding up, so perhaps the given \\binom{23}{4} is incorrect? But I calculated it as 8855, and it matches my calculation.\n\nLet's calculate \\binom{23}{4} directly: \\frac{23×22×21×20}{24} = as above 212520 / 24 = 8855, correct.\n\n\\binom{23}{3} = \\frac{23×22×21}{6} = 10626 / 6 = 1771, correct.\n\nSum 1771 + 8855 = 10556.\n\nBut \\binom{24}{4} = \\frac{24×23×22×21}{24} = 23×22×21 / 1? No, as above, 10626.\n\n24×23×22×21 / 24 = 23×22×21, but that's not divided by the factorial.\n\n4! = 24, so it's 24×23×22×21 / 24 = 23×22×21, which is 10626, but this is not the binomial coefficient because we have to divide by 4! = 24, but in this expression, we have already divided by 24 in the denominator? Let's see the formula.\n\nStandard formula: \\binom{n}{k} = \\frac{n!}{k! (n-k)!}, so for n=24, k=4, \\binom{24}{4} = \\frac{24!}{4! 20!} = \\frac{24×23×22×21×20!}{24 × 20!} = \\frac{24×23×22×21}{24} = 23×22×21 / 1? 24 in numerator and denominator cancel, so it is 23×22×21, but 23×22×21 = 10626, and this is equal to \\binom{24}{4}? But \\binom{24}{4} should be less than 24^4 or something, but 10626 seems reasonable, but let's check with a smaller number.\n\nFor example, \\binom{4}{2} = 6, and \\frac{4×3}{2} = 6, correct. \\binom{5}{2} = 10, \\frac{5×4}{2} = 10, correct.\n\nFor \\binom{24}{4}, it should be \\frac{24×23×22×21}{24} , but 24 in denominator is 4! , which is 24, so yes, \\frac{24×23×22×21}{24} = 23×22×21, but 23×22×21 = 10626, but this is not correct because for example \\binom{4}{2} = \\frac{4×3}{2} = 6, which is correct, but in this case for \\binom{24}{4}, it is \\frac{24×23×22×21}{4×3×2×1} = \\frac{24}{4} × \\frac{23×22×21}{6} = 6 × \\binom{23}{3} = 6 × 1771 = 10626.\n\nBut in the expression \\frac{24×23×22×21}{24}, the 24 in denominator is 4! , but in the calculation, when I say \\frac{24×23×22×21}{24}, this 24 in denominator is actually 4! , but in the value, it's 23×22×21, which is correct, but for \\binom{24}{4}, it is indeed 23×22×21, but 23×22×21 = 10626, and \\binom{24}{4} should be the number of ways, but let's calculate for n=5,k=2: \\binom{5}{2} = 10, and \\frac{5×4}{2} = 10, while \\frac{5×4}{2!} = \\frac{20}{2} = 10, and 2! = 2, so in general \\binom{n}{k} = \\frac{n (n-1) \\cdots (n-k+1)}{k!}.\n\nFor k=4, \\binom{24}{4} = \\frac{24×23×22×21}{4×3×2×1} = \\frac{24}{4} × \\frac{23×22×21}{6} = 6 × \\binom{23}{3} = 6 × 1771 = 10626.\n\nBut according to Pascal, \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4} = 1771 + 8855 = 10556, and 10556 ≠ 10626, so the identity is not satisfied, which is impossible.\n\nUnless I have the identity wrong. Let me recall Pascal's identity.\n\nPascal's identity: \\binom{n}{k} = \\binom{n-1}{k-1} + \\binom{n-1}{k}.\n\nFor n=24, k=4, \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4}, yes.\n\nBut numerically not matching, so perhaps the given \\binom{23}{4} is wrong. But I calculated it as 8855, and it matches.\n\nLet's calculate \\binom{23}{4} again. \\frac{23×22×21×20}{24} = \\frac{23×22×21×20}{24}.\n\n23×22 = 506\n\n506×21 = 506×20 + 506×1 = 10120 + 506 = 10626\n\n10626×20 = 212520\n\n212520 / 24.\n\n24 × 8850 = 24×8000=192000, 24×850=20400, total 212400, as before.\n\n212520 - 212400 = 120, 24×5 = 120, so 8850 + 5 = 8855, correct.\n\n\\binom{23}{3} = \\frac{23×22×21}{6} = \\frac{10626}{6} = 1771, correct.\n\nSum 1771 + 8855 = 10556.\n\nBut \\binom{24}{4} = \\frac{24×23×22×21}{24} = 23×22×21 = 10626, as above.\n\n10626 - 10556 = 70, so difference of 70.\n\nPerhaps I need to use the correct identity. For \\binom{24}{4}, it should be \\binom{23}{3} + \\binom{23}{4}, but let's calculate \\binom{23}{4} again.\n\nAnother way: \\binom{23}{4} = \\binom{22}{3} + \\binom{22}{4}, but I don't have that.\n\nI know that \\binom{n}{k} = \\binom{n}{n-k}, so \\binom{23}{4} = \\binom{23}{19}, but that doesn't help.\n\nPerhaps the given value for \\binom{23}{5} is incorrect, but I calculated it as 33649, and it matched.\n\nLet's calculate \\binom{24}{4} using another method. \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4} should be correct, but it's not, so maybe in this case it's a calculation error in addition.\n\n1771 + 8855.\n\n1771\n+8855\n------\n1+5=6\n7+5=12, write 2, carry 1\n7+8=15, plus 1=16, write 6, carry 1\n1+8=9, plus 1=10, write 0, carry 1\n then 1, so 10556, but it should be 10626, so perhaps the carry is wrong.\n\nWhen I add:\n\nUnits: 1 + 5 = 6\n\nTens: 7 + 5 = 12, write 2, carry 1 to hundreds.\n\nHundreds: 7 + 8 = 15, plus carry 1 = 16, write 6, carry 1 to thousands.\n\nThousands: 1 (from 1771) + 8 (from 8855) = 9, plus carry 1 = 10, so write 0, carry 1 to ten thousands.\n\nTen thousands: 0 + 0 + carry 1 = 1, so 10556.\n\nBut it should be 10626, so for \\binom{24}{4}, it should be 10626, so perhaps the given \\binom{23}{4} is wrong, or \\binom{23}{3}.\n\nPerhaps I have a fundamental mistake.\n\nLet's calculate \\binom{24}{4} directly for a smaller analogous case. Suppose n=5, k=2. \\binom{5}{2} = 10.\n\n\\binom{4}{1} + \\binom{4}{2} = 4 + 6 = 10, correct.\n\n\\binom{4}{1} = 4, \\binom{4}{2} = 6.\n\nNow for n=24, k=4, \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4}.\n\n\\binom{23}{3} = 1771, \\binom{23}{4} = 8855, sum 10556.\n\nBut direct calculation gives 23*22*21 = 10626, which is for \\binom{24}{4}? 23*22*21 = 10626, but \\binom{24}{4} = \\frac{24×23×22×21}{24} = 23×22×21, yes, but 23*22*21 = 10626, but in the sum, I have 10556, so perhaps the direct calculation is wrong.\n\n23*22 = 506, 506*21 = 506*20 = 10120, 506*1 = 506, sum 10626, yes.\n\nBut for \\binom{24}{4}, it is \\frac{24!}{4!20!} = \\frac{24×23×22×21×20!}{24×20!} = \\frac{24×23×22×21}{24} = 23×22×21, yes, 10626.\n\nBut according to Pascal, it should be \\binom{23}{3} + \\binom{23}{4} = 1771 + 8855 = 10556, which is 70 less.\n\nPerhaps the identity is \\binom{n}{k} = \\binom{n-1}{k-1} + \\binom{n-1}{k}, but for k=4, n=24, \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4}, but \\binom{23}{4} is for k=4, n-1=23, yes.\n\nUnless \\binom{23}{4} is not 8855, but the given is 33649 for \\binom{23}{5}, which is larger, so perhaps the addition is correct but the values are for different things.\n\nI need to find \\binom{25}{5}, so perhaps I should use the values given and proceed, even if there's a discrepancy, but that might be risky.\n\nPerhaps I can use the fact that \\binom{25}{5} = \\binom{24}{4} + \\binom{24}{5}, and \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4} = 1771 + 8855 = 10556, and \\binom{24}{5} = \\binom{23}{4} + \\binom{23}{5} = 8855 + 33649 = 42494, then sum 10556 + 42494 = 53050, but from direct calculation, \\binom{25}{5} = 53,130, as I calculated earlier, so 53050 vs 53130, still off by 80.\n\n53130 - 53050 = 80, so close but not the same.\n\nPerhaps I can use the given values to find \\binom{24}{4} and \\binom{24}{5} correctly.\n\nAnother way: perhaps use the identity for \\binom{25}{5} directly.\n\n\\binom{25}{5} = \\binom{24}{4} + \\binom{24}{5}, as I have.\n\nBut \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4} = 1771 + 8855 = 10556\n\n\\binom{24}{5} = \\binom{23}{4} + \\binom{23}{5} = 8855 + 33649 = 42494\n\nSum 10556 + 42494 = 53050\n\nBut direct calculation gave 53130, so perhaps the given \\binom{23}{5} is wrong.\n\nLet me calculate \\binom{23}{5} directly: \\frac{23×22×21×20×19}{120} = as before, 23*22=506, 506*21=10626, 10626*20=212520, 212520*19=4,037,880, then /120 = 4,037,880 / 120.\n\n120 * 33649 = 120*30000=3,600,000; 120*3649.\n\n120*3000=360,000; 120*649=120*600=72,000; 120*49=5,880; so 72,000+5,880=77,880; so total 360,000+77,880=437,880; then total 3,600,000 + 437,880 = 4,037,880, yes, so 33649 is correct.\n\nPerhaps \\binom{23}{4} is not 8855. Let's calculate \\binom{23}{4} again.\n\n\\binom{23}{4} = \\frac{23×22×21×20}{24} = \\frac{23×22×21×20}{24}.\n\n23*22=506\n\n506*21=10,626\n\n10,626*20=212,520\n\n212,520 / 24.\n\n24 * 8855 = as before, 24*8000=192,000; 24*855=24*800=19,200; 24*55=1,320; 19,200+1,320=20,520; total 192,000+20,520=212,520, yes.\n\nBut for \\binom{24}{4} = \\frac{24×23×22×21}{24} = 23×22×21 = 23*22=506, 506*21=10,626, as before.\n\nBut \\binom{24}{4} should also be \\binom{23}{3} + \\binom{23}{4} = 1771 + 8855 = 10,556, but 10,626 ≠ 10,556.\n\nUnless I miscalculated the sum. 1771 + 8855.\n\n1771\n+8855\n- first, 1+5=6\n7+5=12, write 2, carry 1\n7+8=15, plus 1=16, write 6, carry 1\n1+8=9, plus 1=10, so 10,556, but it should be 10,626, so perhaps the carry is correct, but the values are for different indices.\n\nPerhaps the given \\binom{23}{3} is for k=3, but in the sum for \\binom{24}{4}, it should be \\binom{23}{3} and \\binom{23}{4}, but maybe \\binom{23}{4} is given as 8855, but it should be something else.\n\nLet's calculate \\binom{23}{4} using the formula with different order.\n\n\\binom{23}{4} = \\frac{23×22×21×20}{4×3×2×1} = \\frac{23}{1} × \\frac{22}{2} × \\frac{21}{3} × \\frac{20}{4} / 1? Let's see.\n\nUsually, we compute as \\frac{23×22×21×20}{24}.\n\n23/1 = 23, but better to pair.\n\n\\frac{23}{1} * \\frac{22}{2} * \\frac{21}{3} * \\frac{20}{4} = 23 * 11 * 7 * 5.\n\n23*11 = 253\n\n253*7 = 1771\n\n1771*5 = 8855, same as before.\n\nSo correct.\n\nBut for \\binom{24}{4} = \\frac{24}{1} * \\frac{23}{2} * \\frac{22}{3} * \\frac{21}{4} ? No, that's not standard.\n\n\\binom{24}{4} = \\frac{24×23×22×21}{24} = as before.\n\nPerhaps in Pascal's identity, for \\binom{24}{4}, it is \\binom{23}{3} + \\binom{23}{4}, but \\binom{23}{4} is correct, but maybe the sum is not 10556, but let's add: 1771 + 8855.\n\n1771 + 8855: 1700 + 8855 = 10555, then +71 = 10556, yes.\n\nBut 1771 + 8855: 8855 + 1700 = 10555, +71 = 10556.\n\nBut it should be 10626, so difference of 70.\n\n70 is 7*10, so perhaps there's a carry error in the addition, but it's correct.\n\nUnless the given \\binom{23}{3} is not 1771, but the problem says it is.\n\nPerhaps for \\binom{24}{4}, it is not \\binom{23}{3} + \\binom{23}{4}, but \\binom{23}{4} + \\binom{23}{5} or something, but that doesn't make sense.\n\nLet's recall that \\binom{n}{k} = \\binom{n-1}{k-1} + \\binom{n-1}{k}, so for n=24, k=4, \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4}, yes.\n\nPerhaps the given values are for different n, but the problem says for 23.\n\nAnother idea: perhaps \\binom{23}{5} = 33649, but for \\binom{24}{5} = \\binom{23}{4} + \\binom{23}{5} = 8855 + 33649 = 42494, and direct \\binom{24}{5} = \\frac{24×23×22×21×20}{120} = \\frac{24}{24} * 23*22*21*20 / 5? 4! = 24, so \\frac{24×23×22×21×20}{120}.\n\n24/24 =1, but 120/24 =5, so \\frac{24}{24} * \\frac{23×22×21×20}{5} = \\frac{23×22×21×20}{5}.\n\n23*22=506, 506*21=10626, 10626*20=212520, /5 = 42504.\n\nBut I have 42494, so 42504 vs 42494, difference of 10, so close.\n\n42504 - 42494 = 10, so off by 10.\n\nSimilarly for \\binom{24}{4}, I have 10556, should be 10626, off by 70.\n\n70 and 10, not the same.\n\nPerhaps the given \\binom{23}{5} is incorrect. Let's calculate \\binom{24}{5} from direct: \\frac{24×23×22×21×20}{120} = as above, 24*23=552, 552*22=12144, 12144*21=255024, 255024*20=5,100,480, then /120.\n\n5,100,480 / 120 = 5,100,480 / 12 / 10 = 425,040 / 10 = 42,504? 120 = 12*10, so /10 = 510,048, then /12.\n\n510,048 / 12.\n\n12 * 42,504 = 12*40,000=480,000; 12*2,504=12*2,500=30,000; 12*4=48; total 30,000+48=30,048; so 480,000+30,048=510,048, yes, so 42,504.\n\nBut from given, \\binom{24}{5} = \\binom{23}{4} + \\binom{23}{5} = 8855 + 33649 = 42,494, which is 10 less than 42,504.\n\nSimilarly for \\binom{24}{4}, given as 10556, should be 10626, off by 70.\n\nBut 70 for one, 10 for the other, not consistent.\n\nPerhaps there's a typo in the given values, but the problem gives them, so I should use them as given.\n\nPerhaps for \\binom{25}{5}, I can use a different approach.\n\nAnother way: use the identity involving \\binom{23}{k} to find \\binom{25}{5}.\n\nI recall that \\binom{n+2}{k} = \\binom{n}{k-2} + 2\\binom{n}{k-1} + \\binom{n}{k}, or something like that.\n\nFrom Pascal's identity twice.\n\nFor example, \\binom{25}{5} = \\binom{24}{4} + \\binom{24}{5}, as before.\n\nAnd \\binom{24}{4} = \\binom{23}{3} + \\binom{23}{4}\n\n\\binom{24}{5} = \\binom{23}{4} + \\binom{23}{5}\n\nSo \\binom{25}{5} = [ \\binom{23}{3} + \\binom{23}{4} ] + [ \\binom{23}{4} + \\binom{23}{5} ] = \\binom{23}{3} + 2\\binom{23}{4} + \\binom{23}{5}\n\nOh! I see, I forgot that in my initial approach, I had \\binom{24}{4} and \\binom{24}{5}, but when I add them, it's \\binom{23}{3} + \\binom{23}{4} + \\binom{23}{4} + \\binom{23}{5} = \\binom{23}{3} + 2\\binom{23}{4} + \\binom{23}{5}\n\nYes, so I can compute it directly from the given values.\n\nGiven: \\binom{23}{3} = 1771, \\binom{23}{4} = 8855, \\binom{23}{5} = 33649\n\nSo \\binom{25}{5} = 1771 + 2*8855 + 33649\n\nCompute step by step.\n\nFirst, 2*8855 = 17710\n\nThen, 1771 + 17710 = 19481\n\nThen, 19481 + 33649\n\n19481 + 33649\n\nAdd: 19481 + 33000 = 52481, plus 649 = 52481 + 600 = 53081, plus 49 = 53130\n\n19481 + 33649\n\nUnits: 1+9=10, write 0, carry 1\n\nTens: 8+4=12, plus 1=13, write 3, carry 1\n\nHundreds: 4+6=10, plus 1=11, write 1, carry 1\n\nThousands: 9+3=12, plus 1=13, write 3, carry 1\n\nTen thousands: 1+3=4, plus 1=5, so 53130.\n\nYes, so 53130.\n\nNow, this matches my direct calculation for \\binom{25}{5}, which was 53,130.\n\nSo, even though individually the intermediate steps might have discrepancies, but when I use the combination, it works.\n\nIn my initial approach, I computed \\binom{24}{4} and \\binom{24}{5} separately and added, but I must have made a mistake in those computations, but since the problem gives the values, and using the combined expression gives the correct answer, I should use that.\n\nSo, \\binom{25}{5} = \\binom{23}{3} + 2\\binom{23}{4} + \\binom{23}{5} = 1771 + 2*8855 + 33649\n\nNow compute: 2*8855 = 17710\n\n1771 + 17710 = 19481\n\n19481 + 33649 = 53130\n\nAs above.\n\nTo confirm: 1771 + 17710 = 19481? 1771 + 17000 = 18771, plus 710 = 19481, yes.\n\n19481 + 33649: 19000 + 33000 = 52000, 481 + 649 = 1130, total 52000 + 1130 = 53130, yes.\n\nSo the answer is 53130.\n\nI could have computed it directly as well, but since the problem gives the values, this is fine.\n\nSo, \\boxed{53130}\n\nIn the box.\n\nSo, the final answer is 53130."
a_token = tokenizer.tokenize(a)

test_token_segments = []
test_current_segment = []
for token in a_token:
    if "\n\n" in token:
        if test_current_segment:
            test_token_segments.append(test_current_segment)
            test_current_segment = []
    else:
        test_current_segment.append(token)

# add the last segment (if exists)
if test_current_segment:
    test_token_segments.append(test_current_segment)

In [159]:
# test
b = "so:\n\nAlign:\n\n  8855\n\n+33649\n\n------\n\nStart from right:\n\n5+9=14"
tokenizer.tokenize(b)

['so',
 ':\n\n',
 'Align',
 ':\n\n',
 '▁▁',
 '88',
 '55',
 '\n\n',
 '<0x2B>',
 '33',
 '649',
 '\n\n',
 '------\n\n',
 'Start',
 '▁from',
 '▁right',
 ':\n\n',
 '<0x35>',
 '<0x2B>',
 '<0x39>',
 '<0x3D>',
 '14']

In [160]:
tokenizer.eos_token_id

45892

In [135]:
confidences = []  # list len=157
start = 0
for segment in token_segments:
    seg_ids = tokenizer.convert_tokens_to_ids(segment)
    end = start + len(seg_ids)
    if gen_logps.numel() > 0 and end > start and end <= gen_logps.numel():
        seg_logps = gen_logps[start:end]
        conf = _summarize_selected_logprobs(seg_logps, policy='avg2')
        confidences.append(conf)
    elif end > start:
        confidences.append(0.0)
    start = end

In [144]:
 try:
    if not os.path.exists(hidden_save_path):
        full_inputs = tokenizer(full_text, return_tensors="pt")
        save_pangu_think_split_tokens_only(
            model, tokenizer, full_inputs.input_ids.to(device), full_text, hidden_save_path,
            hs_device=args.hs_device
        )
        del full_inputs
    else:
        print(f"[rank {rank}] hidden exists, skip: {hidden_save_path}")
except torch.npu.OutOfMemoryError:
    print(f"[WARN][rank {rank}] hidden save OOM at idx={i}; will rely on rescue next run.")
    _clear_npu()
except Exception as he:
    print(f"[WARN][rank {rank}] hidden save failed at idx={i}: {he} (rescueable next run)")

[rank 0] hidden exists, skip: /home/ma-user/work/dataset/outputs_yulin/test/openPangu-Embedded-7B-V1.1/Math_Math/hidden_000.pt


In [151]:
a = tokenizer("\n\n\nhow are you").input_ids
a

[1, 92421, 43743, 34051, 37667]

In [88]:
end

3404

In [101]:
a = tokenizer(text_before_think, add_special_tokens=False).input_ids
len(a)

3472

In [102]:
len(confidences)

163

In [85]:
len(text_segments)

163

In [65]:
gen_token_ids

tensor([45974,    76, 32240,  ..., 89111,   128, 45892])

In [37]:
output.sequences

tensor([[    1, 45887, 70914,  ..., 89111,   128, 45892]], device='npu:0')

In [48]:
inputs.input_ids

tensor([[     1,  45887,  70914,  89246,  20860,  15578,  27438,  40811,  27438,
             47,  35318,  34701,  31202,  20582,  15299,  14103,  88816,  96190,
          96234,  45892,  45887,  62205,  89246,  45660,  35740,  26882,  43888,
          45728,  42581,  26330,  13873,     47,  31071,  37295,  24686,  41839,
          43744,  23967,  14560,     61,  36682,     47,  27663,     47,  35318,
          30335,  27124,     49,  41528,  35740,  31071,  41141,  35740,  17178,
             61,  88816,  30928,    126,  42611,  34803,  88909,    111,     67,
          89008,    116,  35961,    128,    111,  92136,  88976,  31071,  37295,
          36682,  89028, 118221,  31071,  37295,  36682,  35318,  27663, 145979,
             13,  89182,  31071,  37295,  27663,  89028, 136897,  31071,  37295,
          27663,  35318,  30335,  27124, 145979,     13,  88921,  31071,  37295,
          30335,  27124,  89028, 139034,  31071,  37295,  36682,  35318,  30335,
          27124, 145979,    

In [106]:
tokenizer.decode(
    45974,
    skip_special_tokens=False
)

'[unused16]'